# Part 2A: Zero-shot and Few-shot Baselines

**Steps covered:**
- Step 6: Zero-shot and Few-shot LLM Baselines

This notebook runs only the LLM baseline classification step.


In [5]:
!pip uninstall -y bitsandbytes accelerate transformers
!pip install -q transformers==4.38.0
!pip install accelerate
!pip install -i https://pypi.org/simple/ bitsandbytes

Found existing installation: accelerate 0.27.0
Uninstalling accelerate-0.27.0:
  Successfully uninstalled accelerate-0.27.0
Found existing installation: transformers 4.38.0
Uninstalling transformers-4.38.0:
  Successfully uninstalled transformers-4.38.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.18.1 requires accelerate>=0.21.0, which is not installed.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.0 which is incompatible.
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
Looking in indexes: https://pypi.org/simple/
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)


# Setup & Dependencies

In [1]:
!pip install -q scikit-learn==1.4.0 pandas==2.2.0 numpy==1.26.4 matplotlib==3.8.3 seaborn==0.13.2
!pip install -q datasets==2.17.0 transformers==4.38.0 accelerate==0.27.0
# Install required packages
!pip install -q textstat spacy nltk
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 234.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.feature_selection import f_classif
from sklearn.preprocessing import LabelEncoder, StandardScaler

print("All imports successful.")

All imports successful.


In [3]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/BeyondFK'
    print("Google Drive mounted.")
except:
    BASE_DIR = '.'
    print("Running locally.")

# Directories (must match Part 1)
DATA_DIR    = os.path.join(BASE_DIR, 'data')
STATIC_DIR  = os.path.join(BASE_DIR, 'static_metrics')
PROMPT_DIR  = os.path.join(BASE_DIR, 'prompt_metrics')
OUTPUT_DIR  = os.path.join(BASE_DIR, 'results')
OOD_DIR     = os.path.join(BASE_DIR, 'ood')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(OOD_DIR, exist_ok=True)

print(f"DATA_DIR   : {DATA_DIR}")
print(f"STATIC_DIR : {STATIC_DIR}")
print(f"PROMPT_DIR : {PROMPT_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted.
DATA_DIR   : /content/drive/MyDrive/BeyondFK/data
STATIC_DIR : /content/drive/MyDrive/BeyondFK/static_metrics
PROMPT_DIR : /content/drive/MyDrive/BeyondFK/prompt_metrics
OUTPUT_DIR : /content/drive/MyDrive/BeyondFK/results


# Step 6: Zero-shot & Few-shot LLM Baselines

**Paper Section 4.4**

Instead of using the LLM to answer 63 yes/no questions (prompt-based metrics),
here we ask the LLM to directly classify the difficulty level of the text.

Two settings:
- **Zero-shot**: No examples provided, just the text + label options
- **Few-shot**: 2 examples per class (6 total) provided before the test text

**Note:** This step requires a GPU.

In [4]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Step 6 will be slow. Consider skipping to Step 7.")

GPU available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [5]:
# Load test data
df_test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))

print(f"Test set size: {len(df_test)}")
print(f"Label distribution:\n{df_test['education_level'].value_counts()}")

Test set size: 910
Label distribution:
education_level
middle        304
high          303
elementary    303
Name: count, dtype: int64


In [6]:
# ============================================================
# Gemma-7b-it requires a special chat format with turn tokens.
# Plain text prompts cause ~97/100 invalid responses.
# This function wraps the prompt in Gemma's expected format.
# ============================================================

def build_prompt_gemma(text, mode='zero', df_train=None, n_per_class=2):
    """
    Build prompt in Gemma-7b-it chat format.
    Gemma expects: <start_of_turn>user ... <end_of_turn><start_of_turn>model
    Without this format, Gemma produces ~97% invalid responses.
    """
    if mode == 'zero':
        instruction = (
            "Your task is to predict the education level of a given text.\n"
            "Reply with exactly one of these labels and nothing else:\n"
            "elementary school\n"
            "middle school\n"
            "high school\n\n"
            f"Text: {text[:400]}\n"
            "Educational level:"
        )
    else:
        # Build few-shot examples
        examples = []
        for level in ['elementary', 'middle', 'high']:
            samples = df_train[df_train['education_level'] == level].sample(
                n=n_per_class, random_state=42
            )
            for _, row in samples.iterrows():
                examples.append(
                    f"Text: {str(row['full_text'])[:200]}\nEducational level: {level} school"
                )
        examples_str = "\n\n".join(examples)
        instruction = (
            "Your task is to predict the education level of a given text.\n"
            "Reply with exactly one of these labels and nothing else:\n"
            "elementary school\n"
            "middle school\n"
            "high school\n\n"
            "Examples:\n\n"
            f"{examples_str}\n\n"
            "Now predict:\n"
            f"Text: {text[:400]}\n"
            "Educational level:"
        )

    # Wrap in Gemma chat format
    prompt = f"<start_of_turn>user\n{instruction}<end_of_turn>\n<start_of_turn>model\n"
    return prompt


def build_prompt_generic(text, mode='zero', df_train=None, n_per_class=2):
    """
    Plain text prompt for Mistral / Llama2 (they handle plain text fine).
    """
    if mode == 'zero':
        return (
            "Your task is to predict the education level corresponding to a given text.\n"
            "You are provided with three labels to choose from:\n"
            "1) elementary school\n"
            "2) middle school\n"
            "3) high school\n\n"
            f"Text: {text[:400]}\n"
            "Educational level: "
        )
    else:
        examples = []
        for level in ['elementary', 'middle', 'high']:
            samples = df_train[df_train['education_level'] == level].sample(
                n=n_per_class, random_state=42
            )
            for _, row in samples.iterrows():
                examples.append(
                    f"Text: {str(row['full_text'])[:300]}\nEducational level: {level} school"
                )
        examples_str = "\n\n".join(examples)
        return (
            "Your task is to predict the education level corresponding to a given text.\n"
            "Labels: 1) elementary school  2) middle school  3) high school\n\n"
            f"Examples:\n\n{examples_str}\n\n"
            f"Now predict:\nText: {text[:400]}\n"
            "Educational level: "
        )


def build_prompt(text, mode, model_name, df_train=None):
    """
    Router — picks the right prompt format based on model.
    """
    if 'gemma' in model_name.lower():
        return build_prompt_gemma(text, mode=mode, df_train=df_train)
    else:
        return build_prompt_generic(text, mode=mode, df_train=df_train)


print("Prompt templates defined (with Gemma chat format fix).")

Prompt templates defined (with Gemma chat format fix).


In [7]:
def parse_llm_response(response_text):
    """
    Parse LLM response to extract elementary/middle/high.
    If the response doesn't match any label, default to 'elementary'
    (same as paper Section 4.4).
    """
    response_lower = response_text.lower().strip()

    if 'elementary' in response_lower:
        return 'elementary'
    elif 'middle' in response_lower:
        return 'middle'
    elif 'high' in response_lower:
        return 'high'
    else:
        return 'elementary'  # default for invalid responses


def run_zero_few_shot(model, tokenizer, df_test, df_train, mode='zero',
                      n_samples=100, device='cuda', model_name='gemma_7b'):
    """
    Run zero-shot or few-shot classification.

    Args:
        model: loaded HuggingFace model
        tokenizer: loaded tokenizer
        df_test: test dataframe
        df_train: train dataframe (needed for few-shot examples)
        mode: 'zero' or 'few'
        n_samples: number of test samples to evaluate (paper uses 100)
        device: 'cuda' or 'cpu'
        model_name: used to pick correct prompt format

    Returns:
        predictions list, true labels list, macro-F1 score
    """
    df_sample = df_test.sample(n=min(n_samples, len(df_test)), random_state=42)

    predictions = []
    true_labels = []
    invalid_count = 0

    for _, row in df_sample.iterrows():
        text = str(row['full_text'])[:500]
        true_label = row['education_level']

        # Use model-aware prompt builder
        prompt = build_prompt(text, mode=mode, model_name=model_name, df_train=df_train)

        inputs = tokenizer(
            prompt,
            return_tensors='pt',
            truncation=True,
            max_length=2048
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=15,
                do_sample=False,
                temperature=1.0,         # must be 1.0 when do_sample=False
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        # Decode only the newly generated tokens
        new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True)

        pred = parse_llm_response(response)
        if pred == 'elementary' and 'elementary' not in response.lower():
            invalid_count += 1

        predictions.append(pred)
        true_labels.append(true_label)

    macro_f1 = f1_score(true_labels, predictions, average='macro')
    print(f"  Mode: {mode}-shot | Macro-F1: {macro_f1:.4f} | Invalid responses: {invalid_count}")

    return predictions, true_labels, macro_f1

print("Inference functions defined.")

Inference functions defined.


In [8]:
# ============================================================
# CHANGE THIS to switch between models
# Options: 'llama2_7b', 'llama2_13b', 'mistral_7b', 'gemma_7b'
# ============================================================
CURRENT_MODEL = 'gemma_7b'

MODEL_IDS = {
    'llama2_7b':  'meta-llama/Llama-2-7b-chat-hf',
    'llama2_13b': 'meta-llama/Llama-2-13b-chat-hf',
    'mistral_7b': 'mistralai/Mistral-7B-Instruct-v0.2',
    'gemma_7b':   'google/gemma-7b-it',
}

model_id = MODEL_IDS[CURRENT_MODEL]
print(f"Selected model: {CURRENT_MODEL} ({model_id})")

Selected model: gemma_7b (google/gemma-7b-it)


In [9]:
# Load model with 8-bit quantization (matches paper Appendix B)
# Skip this cell if no GPU available

if torch.cuda.is_available():
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from huggingface_hub import login
    from google.colab import userdata

    # Login to HuggingFace (needed for Llama2)
    try:
        hf_token = userdata.get('HF_TOKEN')
        login(token=hf_token)
        print("HuggingFace login successful.")
    except:
        print("No HF_TOKEN found. Add it in Colab Secrets if needed for gated models.")

    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.float16
    )

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map='auto'
    )
    model.eval()
    print(f"Model loaded: {CURRENT_MODEL}")

    # Run zero-shot
    print("\nRunning zero-shot...")
    zs_preds, zs_true, zs_f1 = run_zero_few_shot(
        model, tokenizer, df_test, df_train,
        mode='zero', n_samples=100, model_name=CURRENT_MODEL
    )

    # Run few-shot
    print("Running few-shot...")
    fs_preds, fs_true, fs_f1 = run_zero_few_shot(
        model, tokenizer, df_test, df_train,
        mode='few', n_samples=100, model_name=CURRENT_MODEL
    )

    # Save results
    BASELINE_DIR = os.path.join(OUTPUT_DIR, 'baselines', 'by_model', CURRENT_MODEL)
    os.makedirs(BASELINE_DIR, exist_ok=True)
    baseline_results = {
        'model': CURRENT_MODEL,
        'zero_shot_f1': zs_f1,
        'few_shot_f1': fs_f1,
        'zero_shot_predictions': zs_preds,
        'few_shot_predictions': fs_preds,
        'true_labels': zs_true
    }
    baseline_path = os.path.join(BASELINE_DIR, f'baseline_{CURRENT_MODEL}.json')
    with open(baseline_path, 'w') as f:
        json.dump(baseline_results, f, indent=2)

    print(f"\nResults saved for {CURRENT_MODEL}: {baseline_path}")

else:
    print("No GPU — skipping Step 6. Using placeholder results for table.")
    # Paper reported results (Table 1) as fallback
    zs_f1 = 0.35
    fs_f1 = 0.47

HuggingFace login successful.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded: gemma_7b

Running zero-shot...
  Mode: zero-shot | Macro-F1: 0.1654 | Invalid responses: 0
Running few-shot...
  Mode: few-shot | Macro-F1: 0.1749 | Invalid responses: 10

Results saved for gemma_7b
